# 01 — Pretrain Hybrid CNN + Transformer on ARC-AGI-2 (Colab Pro)

Runtime → Change runtime type → A100 / L4 (recommended) or T4 (slow but works).
Persists checkpoints to `/content/drive/MyDrive/arc_hybrid_runs/` so a session reset doesn't lose work.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pathlib

REPO_DIR = pathlib.Path('/content/arc_hybrid_repo')
if not REPO_DIR.exists():
    !git clone https://github.com/Nitish05/ARC-AGI.git {REPO_DIR}

%cd /content/arc_hybrid_repo
!pip install -q -r requirements.txt

Cloning into '/content/arc_hybrid_repo'...
remote: Enumerating objects: 51, done.
remote: Counting objects: 100% (51/51), done.
remote: Compressing objects: 100% (42/42), done.
remote: Total 51 (delta 10), reused 49 (delta 8), pack-reused 0 (from 0)
Receiving objects: 100% (51/51), 27.94 KiB | 1.47 MiB/s, done.
Resolving deltas: 100% (10/10), done.
/content/arc_hybrid_repo


In [ ]:
# Pull latest from GitHub (run this whenever you've pushed local edits).
!git -C /content/arc_hybrid_repo pull --ff-only

# Hot-reload so edits to arc_hybrid/*.py take effect on the next cell run
# without a kernel restart. Python 3.12 removed the `imp` module, but Colab's
# bundled autoreload still imports it — shim with importlib.reload first.
import sys, types, importlib
if 'imp' not in sys.modules:
    sys.modules['imp'] = types.ModuleType('imp')
    sys.modules['imp'].reload = importlib.reload
%load_ext autoreload
%autoreload 2

In [3]:
# Download datasets into data/raw/
!mkdir -p data/raw
!git clone --depth 1 https://github.com/arcprize/ARC-AGI-2 data/raw/ARC-AGI-2 || true
!git clone --depth 1 https://github.com/fchollet/ARC-AGI data/raw/ARC-AGI || true
# RE-ARC (optional, large): uncomment the next line to include
# !git clone --depth 1 https://github.com/michaelhodel/re-arc data/raw/re-arc

Cloning into 'data/raw/ARC-AGI-2'...
remote: Enumerating objects: 1130, done.
remote: Counting objects: 100% (1130/1130), done.
remote: Compressing objects: 100% (512/512), done.
remote: Total 1130 (delta 628), reused 1074 (delta 618), pack-reused 0 (from 0)
Receiving objects: 100% (1130/1130), 562.39 KiB | 6.25 MiB/s, done.
Resolving deltas: 100% (628/628), done.
Cloning into 'data/raw/ARC-AGI'...
remote: Enumerating objects: 817, done.
remote: Counting objects: 100% (817/817), done.
remote: Compressing objects: 100% (375/375), done.
remote: Total 817 (delta 493), reused 671 (delta 442), pack-reused 0 (from 0)
Receiving objects: 100% (817/817), 406.11 KiB | 5.34 MiB/s, done.
Resolving deltas: 100% (493/493), done.


In [4]:
# Smoke test: 100 steps, tiny config
!python -m arc_hybrid.train.pretrain --config configs/pretrain_small.yaml --steps 100

loading tasks...
total 0 tasks (filtered to max_grid=30)
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/arc_hybrid_repo/arc_hybrid/train/pretrain.py", line 190, in <module>
    main()
  File "/content/arc_hybrid_repo/arc_hybrid/train/pretrain.py", line 124, in main
    raise RuntimeError("no tasks found; check data paths in config")
RuntimeError: no tasks found; check data paths in config


In [ ]:
# Full pretrain — point output dir at Drive so checkpoints survive a session reset.
import yaml, pathlib
cfg_path = pathlib.Path('configs/pretrain_medium.yaml')
cfg = yaml.safe_load(cfg_path.read_text())
cfg['logging']['out_dir'] = '/content/drive/MyDrive/arc_hybrid_runs/medium_v1'
tmp = pathlib.Path('configs/_active.yaml')
tmp.write_text(yaml.safe_dump(cfg))
!python -m arc_hybrid.train.pretrain --config configs/_active.yaml